# Deploy AutoGluon Model with Custom Inference Image

This notebook deploys the trained AutoGluon model to a SageMaker real-time endpoint using a custom inference image built on top of the AutoGluon DLC.

In [ ]:
import boto3
import json
import tarfile
import tempfile
from datetime import datetime
from pathlib import Path
from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes.shapes import ContainerDefinition, ProductionVariant
from sagemaker.core.helper.session_helper import Session, get_execution_role

# Configuration
REGION = boto3.Session().region_name
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
BUCKET = Session().default_bucket()
S3_PREFIX = "autogluon-tabular"
INSTANCE_TYPE = "ml.m5.xlarge"
INSTANCE_COUNT = 1
TIMESTAMP = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
MODEL_NAME = f"ag-tabular-custom-{TIMESTAMP}"
ENDPOINT_CONFIG_NAME = f"ag-tabular-custom-config-{TIMESTAMP}"
ENDPOINT_NAME = f"ag-tabular-custom-{TIMESTAMP}"

# Custom inference image URI
REPO_NAME = "autogluon-custom"
IMAGE_TAG = "ag150-dlc-upgrade"
image_uri = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{REPO_NAME}:{IMAGE_TAG}-inference"

print(f"Region: {REGION}")
print(f"Account: {ACCOUNT_ID}")
print(f"Bucket: {BUCKET}")
print(f"Model Name: {MODEL_NAME}")
print(f"Endpoint Name: {ENDPOINT_NAME}")
print(f"Image URI: {image_uri}")

## Discover IAM Role

In [ ]:
def get_role(role_arn=None):
    """Discover a SageMaker execution role via IAM paginator.
    Note: get_execution_role() returns the SSO role which doesn't work for model creation.
    """
    if role_arn:
        return role_arn
    iam = boto3.client("iam")
    paginator = iam.get_paginator("list_roles")
    for page in paginator.paginate():
        for r in page["Roles"]:
            name = r["RoleName"]
            if "SageMaker" in name or "sagemaker" in name:
                print(f"Discovered role: {r['Arn']}")
                return r["Arn"]
    raise ValueError("No SageMaker IAM role found. Pass role_arn explicitly.")

role = get_role()
print(f"Execution Role: {role}")

## Locate Model Artifact

Find the most recent training job's model artifact.

In [ ]:
sm_client = boto3.client("sagemaker", region_name=REGION)

# Find most recent training job
training_jobs = sm_client.list_training_jobs(
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=50,
    NameContains="ag-custom-dlc-upgrade",
)

if not training_jobs["TrainingJobSummaries"]:
    raise ValueError("No training jobs found with prefix 'ag-custom-dlc-upgrade'")

latest_job = training_jobs["TrainingJobSummaries"][0]
job_name = latest_job["TrainingJobName"]
job_details = sm_client.describe_training_job(TrainingJobName=job_name)
model_artifact_uri = job_details["ModelArtifacts"]["S3ModelArtifacts"]

print(f"Latest training job: {job_name}")
print(f"Model artifact: {model_artifact_uri}")

## Repackage Model with Inference Script

AutoGluon DLC uses TorchServe, which requires `code/inference.py` in model.tar.gz.
The training script already copied serve.py into the model artifact, so we just need to rename it.

In [ ]:
import sagemaker
import os
import sys

s3_client = boto3.client("s3")
sess = sagemaker.Session()

# Download model artifact
with tempfile.TemporaryDirectory() as tmpdir:
    local_model_path = Path(tmpdir) / "model.tar.gz"
    
    bucket_name = model_artifact_uri.split("/")[2]
    key = "/".join(model_artifact_uri.split("/")[3:])
    s3_client.download_file(bucket_name, key, str(local_model_path))
    print(f"Downloaded model to {local_model_path}")
    
    # Extract
    extract_dir = Path(tmpdir) / "model"
    extract_dir.mkdir()
    
    # Handle Python 3.12+ tarfile requirement
    if sys.version_info >= (3, 12):
        with tarfile.open(local_model_path, "r:gz") as tar:
            tar.extractall(extract_dir, filter="data")
    else:
        with tarfile.open(local_model_path, "r:gz") as tar:
            tar.extractall(extract_dir)
    
    print(f"Extracted to {extract_dir}")
    
    # Rename serve.py to inference.py if it exists
    code_dir = extract_dir / "code"
    if (code_dir / "serve.py").exists():
        (code_dir / "serve.py").rename(code_dir / "inference.py")
        print("Renamed serve.py to inference.py")
    elif not (code_dir / "inference.py").exists():
        # If neither exists, copy from local
        code_dir.mkdir(exist_ok=True)
        import shutil
        shutil.copy("serve.py", code_dir / "inference.py")
        print("Copied local serve.py to inference.py")
    
    print(f"Code dir contents: {list(code_dir.iterdir())}")
    
    # Repackage
    repackaged_path = Path(tmpdir) / "model-repackaged.tar.gz"
    with tarfile.open(repackaged_path, "w:gz") as tar:
        for item in extract_dir.iterdir():
            tar.add(item, arcname=item.name)
    
    print(f"Repackaged model to {repackaged_path}")
    
    # Upload repackaged model
    repackaged_s3_uri = sess.upload_data(
        path=str(repackaged_path),
        bucket=BUCKET,
        key_prefix=f"{S3_PREFIX}/custom-image/models",
    )
    print(f"Repackaged model uploaded to: {repackaged_s3_uri}")

model_data_uri = repackaged_s3_uri

## Create Model

In [ ]:
model = Model.create(
    model_name=MODEL_NAME,
    execution_role_arn=role,
    primary_container=ContainerDefinition(
        image=image_uri,
        model_data_url=model_data_uri,
    ),
)

print(f"Model created: {model.model_name}")
print(f"Model ARN: {model.arn}")

## Create Endpoint Configuration

In [ ]:
endpoint_config = EndpointConfig.create(
    endpoint_config_name=ENDPOINT_CONFIG_NAME,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=MODEL_NAME,
            instance_type=INSTANCE_TYPE,
            initial_instance_count=INSTANCE_COUNT,
        )
    ],
)

print(f"Endpoint config created: {endpoint_config.endpoint_config_name}")

## Create and Deploy Endpoint

In [ ]:
endpoint = Endpoint.create(
    endpoint_name=ENDPOINT_NAME,
    endpoint_config_name=ENDPOINT_CONFIG_NAME,
)

print(f"Endpoint created: {endpoint.endpoint_name}")
print(f"Waiting for endpoint to be in service...")

endpoint.wait_for_status("InService")

print(f"Endpoint status: {endpoint.endpoint_status}")
print(f"Endpoint ARN: {endpoint.arn}")

## Test Inference

Send a sample prediction request to the endpoint.

In [ ]:
import pandas as pd

# Sample test data (Adult income dataset - same as training data)
sample_data = pd.DataFrame({
    "age": [39, 52],
    "workclass": ["State-gov", "Self-emp-not-inc"],
    "fnlwgt": [77516, 209642],
    "education": ["Bachelors", "Bachelors"],
    "education-num": [13, 13],
    "marital-status": ["Never-married", "Married-civ-spouse"],
    "occupation": ["Adm-clerical", "Exec-managerial"],
    "relationship": ["Not-in-family", "Husband"],
    "race": ["White", "White"],
    "sex": ["Male", "Male"],
    "capital-gain": [2174, 0],
    "capital-loss": [0, 0],
    "hours-per-week": [40, 45],
    "native-country": ["United-States", "United-States"],
})
csv_payload = sample_data.to_csv(index=False)

response = endpoint.invoke(
    body=csv_payload,
    content_type="text/csv",
    accept="application/json",
)

# Response body is a botocore StreamingBody
predictions = json.loads(response.body.read().decode("utf-8"))
print("\nPredictions:")
print(json.dumps(predictions, indent=2))

## Cleanup (Optional)

Delete the endpoint when done to avoid charges.

In [ ]:
# Uncomment to delete endpoint
# endpoint.delete()
# print(f"Deleted endpoint: {ENDPOINT_NAME}")